# RetailPulse AI: Classical Statistical Time-Series Benchmark (SARIMAX)
**Academic Level:** 2nd Year 2nd Semester Undergraduate Data Science Project  
**Focus:** Single-Store Seasonal ARIMA with Exogenous Variables Benchmark

## 1. Objective & Econometric Background
Classical time-series forecasting relies heavily on the **Box-Jenkins methodology (ARIMA)**. When seasonal patterns and external drivers (exogenous regressors) are present, **SARIMAX** is the standard benchmark.

In this notebook, we fit a SARIMAX $(1, 1, 1) \times (1, 0, 0)_{52}$ model with external regressors (`Holiday_Flag`, `Fuel_Price`, `CPI`) on Store #1 to evaluate how traditional statistical models compare to modern ML ensembles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Load data
df = pd.read_csv('../data/Walmart_Store_sales.csv')
df.columns = [c.strip().replace('\\_', '_') for c in df.columns]
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Filter Store #1 and set weekly 7D frequency
store_1 = df[df['Store'] == 1].sort_values('Date').set_index('Date')
sales_series = store_1['Weekly_Sales'].asfreq('7D')
exog_vars = store_1[['Holiday_Flag', 'Fuel_Price', 'CPI']].asfreq('7D')

# Split last 12 weeks for testing
weeks = 12
train_y, test_y = sales_series.iloc[:-weeks], sales_series.iloc[-weeks:]
train_exog, test_exog = exog_vars.iloc[:-weeks], exog_vars.iloc[-weeks:]

print(f"Training weeks: {len(train_y)}, Test weeks: {len(test_y)}")

## 2. Fit SARIMAX Model
We fit a seasonal autoregressive specification with annual period $s=52$ and exogenous inputs.

In [ ]:
model = SARIMAX(
    train_y,
    exog=train_exog,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 0, 52),
    enforce_stationarity=False,
    enforce_invertibility=False
)

results = model.fit(disp=False, maxiter=100)
print(results.summary())

## 3. Forecast & Evaluate on Holdout Period
Generating out-of-sample forecast along with 95% confidence intervals.

In [ ]:
forecast_res = results.get_forecast(steps=weeks, exog=test_exog)
pred_mean = forecast_res.predicted_mean
conf_int = forecast_res.conf_int(alpha=0.05)

sarimax_rmse = np.sqrt(np.mean((test_y.values - pred_mean.values)**2))
sarimax_mae = np.mean(np.abs(test_y.values - pred_mean.values))
print(f"SARIMAX Test RMSE: ${sarimax_rmse:,.2f}")
print(f"SARIMAX Test MAE:  ${sarimax_mae:,.2f}")

# Visualization
plt.figure(figsize=(14, 5))
plt.plot(train_y.index[-24:], train_y.iloc[-24:], label='Recent History (Train)', color='#334155')
plt.plot(test_y.index, test_y.values, label='Actual Holdout Sales', color='#0f172a', marker='o')
plt.plot(pred_mean.index, pred_mean.values, label='SARIMAX Forecast', color='#0d9488', linestyle='--', marker='s')
plt.fill_between(conf_int.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='#0d9488', alpha=0.15, label='95% Confidence Interval')
plt.title(f'Store #1: SARIMAX 12-Week Out-of-Sample Forecast (RMSE: ${sarimax_rmse:,.2f})', fontsize=13, fontweight='bold')
plt.ylabel('Weekly Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Key Comparative Takeaways
1. **Statistical Strengths:** SARIMAX naturally provides analytical confidence intervals based on theoretical error distributions.
2. **Scalability Limitations:** SARIMAX must be fitted individually to each store, making it computationally intensive and challenging to maintain across large retail networks (45+ stores).
3. **Global Machine Learning Advantage:** A unified tree-based ensemble (such as Gradient Boosting or Random Forest) learns cross-store patterns simultaneously while running in milliseconds during inference.